# MIST 5400 Week 11 Lab, Part 1: from a trained model to a web app

This notebook follows Part 1 of the handout. Cells 1 to 4 are the week 3 churn model, unchanged.
Cells 5 and 6 are the two steps from the handout: ask Gemini for a Gradio cell, then run it and share the link.

Cell 1 pulls the dataset from the course GitHub repository (the week11 folder only), so there is nothing to upload.

## The model (week 3, unchanged)

In [ ]:
# Pull only the week11 folder of Pearl's GitHub repository, which has the dataset.
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/pearl-yu/mist5400fall2026.git
%cd mist5400fall2026
!git sparse-checkout set week11
%cd week11
!ls

In [ ]:
import pandas as pd

df = pd.read_csv("churn_data.csv")
print(df.shape)
df.head()

In [ ]:
# one hot encode every text column, drop one level each so the columns are not redundant
df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop("Churn_Yes", axis=1)
y = df_encoded["Churn_Yes"]
print(X.shape, "features after encoding")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_reg_model = LogisticRegression(max_iter=1000)
log_reg_model.fit(X_train, y_train)

print("test accuracy:", round(accuracy_score(y_test, log_reg_model.predict(X_test)), 3))

## Step 1: ask Gemini for the Gradio cell

Click the sparkle (Gemini) button in a **new cell below this one** and paste the prompt from the handout.
For the churn model the brackets are already filled in:

> This notebook trains a logistic regression called `log_reg_model` on `churn_data.csv` after `pd.get_dummies(df, drop_first=True)`, with the training features in `X_train`.
>
> Write a Gradio app as one code cell for this notebook that:
>
> - uses the trained model and X_train.columns already in memory (do not retrain)
> - has a function taking tenure, MonthlyCharges, Contract, InternetService, PaymentMethod, PaperlessBilling, SeniorCitizen, TechSupport and a decision threshold; the other raw columns are held at typical values inside the function
> - builds one raw row with the original column names, applies get_dummies, then reindexes to X_train.columns with fill_value=0
> - returns the probability of churn and a decision at the threshold
> - uses gr.Interface with a slider, dropdown or checkbox per input, live=True, a title, and a description paragraph marked TODO that I will write
> - ends with demo.launch(share=True)
>
> Start the cell with !pip install -q gradio. Then list every default you chose that I did not specify.

Gemini writes the cell for you. The cell below is what it should look like; use it to compare, or run it directly if Gemini is unavailable.

Before running, check the three lines marked `# CHECK`: the raw row uses the original column names, `get_dummies` is followed by `reindex(columns=X_train.columns, fill_value=0)`, and the prediction calls `log_reg_model`.

In [ ]:
!pip install -q gradio
import gradio as gr
import pandas as pd

# model (log_reg_model) and X_train come from the cells above; nothing to load

def score(tenure, monthly, contract, internet, payment, paperless, senior, techsup, threshold):
    # one raw row, same column names as churn_data.csv; columns not shown take typical values
    raw = pd.DataFrame([{                                                              # CHECK
        "gender": "Female", "SeniorCitizen": int(senior), "Partner": "No", "Dependents": "No",
        "tenure": tenure, "PhoneService": "Yes", "MultipleLines": "No",
        "InternetService": internet,
        "OnlineSecurity": "No" if internet != "No" else "No internet service",
        "OnlineBackup": "No" if internet != "No" else "No internet service",
        "DeviceProtection": "No" if internet != "No" else "No internet service",
        "TechSupport": ("Yes" if techsup else "No") if internet != "No" else "No internet service",
        "StreamingTV": "No" if internet != "No" else "No internet service",
        "StreamingMovies": "No" if internet != "No" else "No internet service",
        "Contract": contract, "PaperlessBilling": "Yes" if paperless else "No",
        "PaymentMethod": payment, "MonthlyCharges": monthly,
    }])
    # same preprocessing as the notebook, then force the exact training columns
    row = pd.get_dummies(raw).reindex(columns=X_train.columns, fill_value=0)              # CHECK
    p = float(log_reg_model.predict_proba(row)[0, 1])                                    # CHECK
    decision = "Likely to churn" if p >= threshold else "Likely to stay"
    return f"{p:.1%}", decision

demo = gr.Interface(
    fn=score,
    inputs=[
        gr.Slider(0, 72, value=12, step=1, label="Tenure (months)"),
        gr.Slider(18, 120, value=70, step=0.5, label="Monthly charges ($)"),
        gr.Dropdown(["Month-to-month", "One year", "Two year"], value="Month-to-month", label="Contract"),
        gr.Dropdown(["DSL", "Fiber optic", "No"], value="Fiber optic", label="Internet service"),
        gr.Dropdown(["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"],
                    value="Electronic check", label="Payment method"),
        gr.Checkbox(value=True, label="Paperless billing"),
        gr.Checkbox(value=False, label="Senior citizen"),
        gr.Checkbox(value=False, label="Has tech support"),
        gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="Decision threshold"),
    ],
    outputs=[gr.Textbox(label="Probability of churn"), gr.Textbox(label="Decision")],
    title="Churn Risk Scorer",
    description="TODO: one paragraph on what this predicts, what data it was trained on, and what the inputs mean.",
    live=True,
)

demo.launch(share=True)

## Step 2: run it and share the link

Running the cell above prints two links. The one ending in `.gradio.live` is public: send it to a teammate's phone and move a slider.

- The link works while this notebook is open and the cell is running; Gradio switches it off after 72 hours. Rerun the cell before the final presentation.
- To change the app, edit the cell (or ask Gemini to change it) and run it again. Describe your changes in the prompt as design choices: which inputs the user can set, what the output should say, what the title and description are.
- For the permanent link due Dec 1, see "How long the link lives" in the handout.

## Your turn

Replace cells 1 to 4 with your own project's data and model, then redo Step 1 with the brackets in the prompt filled in for your model.